In [2]:
import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
import pykitti

from pathlib import Path

from src.proxies.gps_proxy import extract_all_gps_proxies
from src.proxies.imu_proxy import extract_all_imu_proxies
from src.proxies.camera_proxy import extract_all_camera_proxies
from src.proxies.lidar_proxy import extract_all_lidar_proxies

from src.attacks.gps_attack import gps_speed_spoof
from src.attacks.imu_attack import attack_imu_proxies

from src.features.normalization import MotionNormalizer
from src.features.f1_kinematic import extract_all_f1_features
from src.features.f2_scene import compute_f2
from src.features.gmis import compute_gmis

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
base = Path.home() / "SensorTrust" / "datasets" / "kitti"

data = pykitti.raw(
    base_path=str(base),
    date="2011_09_26",
    drive="0009"
)

print(len(data.oxts))

447


In [4]:
gps_clean = extract_all_gps_proxies(data.oxts)

imu_clean = extract_all_imu_proxies(
    data.oxts,
    dt=0.1035
)

camera_frames = [np.array(f) for f in list(data.cam2)]

camera_clean = extract_all_camera_proxies(
    camera_frames
)

velo_scans = list(data.velo)

lidar_clean = extract_all_lidar_proxies(
    velo_scans,
    data.oxts[:len(velo_scans)]
)

Processing scan 20/443
Processing scan 40/443
Processing scan 60/443
Processing scan 80/443
Processing scan 100/443
Processing scan 120/443
Processing scan 140/443
Processing scan 160/443
Processing scan 180/443
Processing scan 200/443
Processing scan 220/443
Processing scan 240/443
Processing scan 260/443
Processing scan 280/443
Processing scan 300/443
Processing scan 320/443
Processing scan 340/443
Processing scan 360/443
Processing scan 380/443
Processing scan 400/443
Processing scan 420/443
Processing scan 440/443


In [5]:
normalizer = MotionNormalizer()

normalizer.fit(
    gps_clean,
    imu_clean,
    camera_clean,
    lidar_clean
)

z_clean = normalizer.transform(
    gps_clean,
    imu_clean,
    camera_clean,
    lidar_clean
)

Normalization parameters fitted:
  gps_delta_v         : μ= -0.1212, σ=  0.4548
  gps_heading_rate    : μ=  0.0423, σ=  1.7530
  imu_delta_v         : μ= -0.1656, σ=  0.4658
  imu_yaw_rate        : μ=  0.0279, σ=  0.1016
  camera_flow         : μ= 11.7695, σ=  5.5256
  lidar_icp           : μ=  0.1131, σ=  0.0318


In [6]:
f1_clean = extract_all_f1_features(
    z=z_clean,
    gps_speed=gps_clean["speed"]
)["f1"]

In [7]:
min_len = min(
    len(z_clean["gps_delta_v"]),
    len(z_clean["lidar_icp"])
)

f2_clean = compute_f2(
    z_clean["gps_delta_v"][:min_len],
    z_clean["lidar_icp"][:min_len]
)

In [8]:
min_len = min(
    len(z_clean["gps_delta_v"]),
    len(z_clean["imu_delta_v"]),
    len(z_clean["lidar_icp"]),
    len(z_clean["camera_flow"])
)

gmis_clean = compute_gmis(
    z_clean["gps_delta_v"][:min_len],
    z_clean["imu_delta_v"][:min_len],
    z_clean["lidar_icp"][:min_len],
    z_clean["camera_flow"][:min_len]
)

In [9]:
normalizer = MotionNormalizer()

normalizer.fit(
    gps_clean,
    imu_clean,
    camera_clean,
    lidar_clean
)

z_clean = normalizer.transform(
    gps_clean,
    imu_clean,
    camera_clean,
    lidar_clean
)

Normalization parameters fitted:
  gps_delta_v         : μ= -0.1212, σ=  0.4548
  gps_heading_rate    : μ=  0.0423, σ=  1.7530
  imu_delta_v         : μ= -0.1656, σ=  0.4658
  imu_yaw_rate        : μ=  0.0279, σ=  0.1016
  camera_flow         : μ= 11.7695, σ=  5.5256
  lidar_icp           : μ=  0.1131, σ=  0.0318


GPS + IMU ATTACK

In [10]:
'''create attacks'''
gps_attacked, labels = gps_speed_spoof(
    data.oxts,
    start_frame=200,
    speed_offset=5.0,
    duration=50
)

imu_attacked = attack_imu_proxies(
    imu_clean,
    attack_type="bias",
    bias=0.5
)

In [11]:
'''gps proxies'''
gps_attack = extract_all_gps_proxies(
    gps_attacked,
    dt=0.1035
)

In [12]:
'''normalize'''
z_attack = normalizer.transform(
    gps_attack,
    imu_attacked,
    camera_clean,
    lidar_clean
)

In [13]:
f1_attack = extract_all_f1_features(
    z=z_attack,
    gps_speed=gps_attack["speed"]
)["f1"]

In [14]:
min_len = min(
    len(z_attack["gps_delta_v"]),
    len(z_attack["lidar_icp"])
)

f2_attack = compute_f2(
    z_attack["gps_delta_v"][:min_len],
    z_attack["lidar_icp"][:min_len]
)

In [15]:
min_len = min(
    len(z_attack["gps_delta_v"]),
    len(z_attack["imu_delta_v"]),
    len(z_attack["lidar_icp"]),
    len(z_attack["camera_flow"])
)

gmis_attack = compute_gmis(
    z_attack["gps_delta_v"][:min_len],
    z_attack["imu_delta_v"][:min_len],
    z_attack["lidar_icp"][:min_len],
    z_attack["camera_flow"][:min_len]
)

In [16]:
print("\n=== GPS + IMU ATTACK ===")

print(f"F1 Clean    : {np.nanmean(f1_clean):.4f}")
print(f"F1 Attack   : {np.nanmean(f1_attack):.4f}")

print()

print(f"F2 Clean    : {np.nanmean(f2_clean):.4f}")
print(f"F2 Attack   : {np.nanmean(f2_attack):.4f}")

print()

print(f"GMIS Clean  : {np.nanmean(gmis_clean):.4f}")
print(f"GMIS Attack : {np.nanmean(gmis_attack):.4f}")


=== GPS + IMU ATTACK ===
F1 Clean    : 0.5598
F1 Attack   : 5.4609

F2 Clean    : 0.9053
F2 Attack   : 1.1468

GMIS Clean  : 0.8221
GMIS Attack : 1.3068


In [17]:
np.save("../src/models/gps_imu_f1.npy", f1_attack)
np.save("../src/models/gps_imu_f2.npy", f2_attack)
np.save("../src/models/gps_imu_gmis.npy", gmis_attack)

GPS + LIDAR

In [18]:
from src.attacks.lidar_attack import inject_phantom_sequence

gps_attacked, _ = gps_speed_spoof(
    data.oxts,
    start_frame=200,
    speed_offset=5.0,
    duration=50
)

lidar_attacked, _ = inject_phantom_sequence(
    velo_scans,
    start_frame=200,
    n_points=20000,
    distance=3.0,
    duration=50
)

IMU + CAMERA

In [19]:
from src.attacks.camera_attack import (
    gaussian_noise as cam_gaussian_noise
)
camera_frames = [np.array(f) for f in list(data.cam2)]

frames_gaussian = [
    cam_gaussian_noise(frame, std=15)
    for frame in camera_frames
] #??????????????

In [20]:
imu_attacked = attack_imu_proxies(
    imu_clean,
    attack_type="bias",
    bias=0.5
)

camera_frames = [np.array(f) for f in list(data.cam2)]

frames_gaussian = [
    cam_gaussian_noise(frame, std=15)
    for frame in camera_frames
]

In [21]:
camera_attack = extract_all_camera_proxies(
    frames_gaussian
)

In [22]:
'''normalize'''
z_attack = normalizer.transform(
    gps_clean,      # CLEAN GPS
    imu_attacked,   # ATTACKED IMU
    camera_attack,  # ATTACKED CAMERA
    lidar_clean     # CLEAN LIDAR
)

In [23]:
f1_attack = extract_all_f1_features(
    z=z_attack,
    gps_speed=gps_clean["speed"]
)["f1"]

In [24]:
min_len = min(
    len(z_attack["gps_delta_v"]),
    len(z_attack["lidar_icp"])
)

f2_attack = compute_f2(
    z_attack["gps_delta_v"][:min_len],
    z_attack["lidar_icp"][:min_len]
)

In [25]:
min_len = min(
    len(z_attack["gps_delta_v"]),
    len(z_attack["imu_delta_v"]),
    len(z_attack["lidar_icp"]),
    len(z_attack["camera_flow"])
)

gmis_attack = compute_gmis(
    z_attack["gps_delta_v"][:min_len],
    z_attack["imu_delta_v"][:min_len],
    z_attack["lidar_icp"][:min_len],
    z_attack["camera_flow"][:min_len]
)

In [26]:
print("\n=== CAMERA + IMU ATTACK ===")

print(f"F1 Clean    : {np.nanmean(f1_clean):.4f}")
print(f"F1 Attack   : {np.nanmean(f1_attack):.4f}")

print()

print(f"F2 Clean    : {np.nanmean(f2_clean):.4f}")
print(f"F2 Attack   : {np.nanmean(f2_attack):.4f}")

print()

print(f"GMIS Clean  : {np.nanmean(gmis_clean):.4f}")
print(f"GMIS Attack : {np.nanmean(gmis_attack):.4f}")


=== CAMERA + IMU ATTACK ===
F1 Clean    : 0.5598
F1 Attack   : 5.1456

F2 Clean    : 0.9053
F2 Attack   : 0.9053

GMIS Clean  : 0.8221
GMIS Attack : 1.1458


In [27]:
np.save("../src/models/camera_imu_f1.npy", f1_attack)
np.save("../src/models/camera_imu_f2.npy", f2_attack)
np.save("../src/models/camera_imu_gmis.npy", gmis_attack)

GPS LIDAR

In [28]:
gps_attacked, labels = gps_speed_spoof(
    data.oxts,
    start_frame=200,
    speed_offset=5.0,
    duration=50
)

lidar_attacked, _ = inject_phantom_sequence(
    velo_scans,
    start_frame=200,
    n_points=20000,
    distance=3.0,
    duration=50
)

In [29]:
gps_attack = extract_all_gps_proxies(
    gps_attacked,
    dt=0.1035
)

In [30]:
lidar_attack = extract_all_lidar_proxies(
    lidar_attacked,
    data.oxts[:len(lidar_attacked)]
)

Processing scan 20/443
Processing scan 40/443
Processing scan 60/443
Processing scan 80/443
Processing scan 100/443
Processing scan 120/443
Processing scan 140/443
Processing scan 160/443
Processing scan 180/443
Processing scan 200/443
Processing scan 220/443
Processing scan 240/443
Processing scan 260/443
Processing scan 280/443
Processing scan 300/443
Processing scan 320/443
Processing scan 340/443
Processing scan 360/443
Processing scan 380/443
Processing scan 400/443
Processing scan 420/443
Processing scan 440/443


In [31]:
z_attack = normalizer.transform(
    gps_attack,
    imu_clean,
    camera_clean,
    lidar_attack
)

In [32]:
f1_attack = extract_all_f1_features(
    z=z_attack,
    gps_speed=gps_attack["speed"]
)["f1"]

In [33]:
min_len = min(
    len(z_attack["gps_delta_v"]),
    len(z_attack["lidar_icp"])
)

f2_attack = compute_f2(
    z_attack["gps_delta_v"][:min_len],
    z_attack["lidar_icp"][:min_len]
)

In [34]:
min_len = min(
    len(z_attack["gps_delta_v"]),
    len(z_attack["imu_delta_v"]),
    len(z_attack["lidar_icp"]),
    len(z_attack["camera_flow"])
)

gmis_attack = compute_gmis(
    z_attack["gps_delta_v"][:min_len],
    z_attack["imu_delta_v"][:min_len],
    z_attack["lidar_icp"][:min_len],
    z_attack["camera_flow"][:min_len]
)

In [35]:
print("\n=== GPS + LIDAR ATTACK ===")

print(f"F1 Clean    : {np.nanmean(f1_clean):.4f}")
print(f"F1 Attack   : {np.nanmean(f1_attack):.4f}")

print()

print(f"F2 Clean    : {np.nanmean(f2_clean):.4f}")
print(f"F2 Attack   : {np.nanmean(f2_attack):.4f}")

print()

print(f"GMIS Clean  : {np.nanmean(gmis_clean):.4f}")
print(f"GMIS Attack : {np.nanmean(gmis_attack):.4f}")


=== GPS + LIDAR ATTACK ===
F1 Clean    : 0.5598
F1 Attack   : 0.8319

F2 Clean    : 0.9053
F2 Attack   : 1.1425

GMIS Clean  : 0.8221
GMIS Attack : 1.0061


In [36]:
np.save("../src/models/gps_lidar_f1.npy", f1_attack)
np.save("../src/models/gps_lidar_f2.npy", f2_attack)
np.save("../src/models/gps_lidar_gmis.npy", gmis_attack)

GPS + IMU + Camera + LiDAR